In [5]:
from google.colab import files

uploaded = files.upload()

Saving Q3_final_outputs.zip to Q3_final_outputs.zip


In [6]:
import os

print(os.listdir("/content"))

['.config', 'garment_03.jpg', 'garment_01.jpg', 'garment_04.jpg', 'q3_outputs', 'garment_02.jpg', 'Q3_final_outputs.zip', 'garment_05.jpg', 'sample_data']


In [7]:
import zipfile
import os

zip_path = "/content/Q3_final_outputs.zip"
Q3_DIR = "/content/q3_outputs"

os.makedirs(Q3_DIR, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(Q3_DIR)

print("Q3 outputs extracted:")
print(os.listdir(Q3_DIR))

Q3 outputs extracted:
['pair_05.png', 'pair_01.png', 'pair_04.png', 'pair_03.png', 'pair_02.png']


In [8]:
!pip install -q transformers accelerate

In [9]:
import torch
from transformers import AutoImageProcessor, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "facebook/dinov2-base"

processor = AutoImageProcessor.from_pretrained(model_name)
dinov2 = AutoModel.from_pretrained(model_name).to(device)
dinov2.eval()

print("DINOv2 loaded successfully ✅")
print("Device:", device)

preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

DINOv2 loaded successfully ✅
Device: cuda


In [10]:
import zipfile
import os

zip_path = "/content/Q3_final_outputs.zip"
Q3_DIR = "/content/q3_outputs"

# Create output folder
os.makedirs(Q3_DIR, exist_ok=True)

# Extract ZIP
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(Q3_DIR)

# Show exactly what was extracted
print("Extracted files:")
for root, dirs, files in os.walk(Q3_DIR):
    for file in files:
        print(os.path.join(root, file))

Extracted files:
/content/q3_outputs/pair_05.png
/content/q3_outputs/pair_01.png
/content/q3_outputs/pair_04.png
/content/q3_outputs/pair_03.png
/content/q3_outputs/pair_02.png


In [11]:
from PIL import Image
import os

for i in range(1, 6):
    path = f"/content/q3_outputs/pair_{i:02d}.png"
    img = Image.open(path)
    print(f"pair_{i:02d}: {img.size}")

pair_01: (384, 512)
pair_02: (384, 512)
pair_03: (384, 512)
pair_04: (384, 512)
pair_05: (384, 512)


In [12]:
from google.colab import files

uploaded = files.upload()

Saving garment_01.jpg to garment_01 (1).jpg
Saving garment_02.jpg to garment_02 (1).jpg
Saving garment_03.jpg to garment_03 (1).jpg
Saving garment_04.jpg to garment_04 (1).jpg
Saving garment_05.jpg to garment_05 (1).jpg


In [13]:
import torch
import torch.nn.functional as F
from PIL import Image
import os

def get_embedding(image):

    inputs = processor(images=image, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = dinov2(**inputs)

    # CLS token embedding
    embedding = outputs.last_hidden_state[:, 0, :]
    embedding = F.normalize(embedding, p=2, dim=1)

    return embedding


def get_upper_body_crop(image):
    """
    Approximate garment region:
    removes most of the head and lower body.
    """
    width, height = image.size

    left = int(width * 0.15)
    right = int(width * 0.85)
    top = int(height * 0.20)
    bottom = int(height * 0.75)

    return image.crop((left, top, right, bottom))


results = []

for i in range(1, 6):

    output_path = f"/content/q3_outputs/pair_{i:02d}.png"
    garment_path = f"/content/garment_{i:02d}.jpg"

    output_image = Image.open(output_path).convert("RGB")
    garment_image = Image.open(garment_path).convert("RGB")

    # Crop approximate garment area from generated result
    garment_region = get_upper_body_crop(output_image)

    # Generate embeddings
    generated_embedding = get_embedding(garment_region)
    original_embedding = get_embedding(garment_image)

    # Cosine similarity
    similarity = F.cosine_similarity(
        generated_embedding,
        original_embedding
    ).item()

    results.append({
        "pair_id": f"pair_{i:02d}",
        "garment_fidelity": similarity
    })

    print(
        f"pair_{i:02d}: "
        f"garment fidelity = {similarity:.4f}"
    )

pair_01: garment fidelity = 0.3398
pair_02: garment fidelity = 0.2471
pair_03: garment fidelity = 0.3830
pair_04: garment fidelity = 0.3216
pair_05: garment fidelity = 0.3819


In [14]:
from google.colab import files

uploaded = files.upload()

Saving person_01.png to person_01.png
Saving person_02.png to person_02.png
Saving person_03.png to person_03.png
Saving person_04.png to person_04.png
Saving person_05.png to person_05.png


In [15]:
!pip install -q insightface onnxruntime-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 762.2/762.2 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.8/249.8 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 91.3 MB/s eta 0:00:00


In [16]:
from insightface.app import FaceAnalysis
import cv2
import numpy as np

app = FaceAnalysis(
    name="buffalo_l",
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
)

app.prepare(ctx_id=0, det_size=(640, 640))

print("InsightFace loaded successfully ✅")

download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:07<00:00, 35389.33KB/s]
/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:153: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)
InsightFace 

In [17]:
import cv2

img = cv2.imread("/content/person_01.png")

faces = app.get(img)

print("Faces detected:", len(faces))

for i, face in enumerate(faces):
    print(
        f"Face {i+1}: "
        f"bbox={face.bbox.astype(int).tolist()}, "
        f"embedding size={face.embedding.shape}"
    )

Faces detected: 1
Face 1: bbox=[337, 84, 473, 254], embedding size=(512,)


In [18]:
import cv2
import numpy as np

def get_face_embedding(image_path):
    img = cv2.imread(image_path)

    if img is None:
        return None

    faces = app.get(img)

    if len(faces) == 0:
        return None

    # Select the largest detected face
    face = max(
        faces,
        key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1])
    )

    embedding = face.embedding.astype(np.float32)

    # Normalize embedding
    embedding = embedding / np.linalg.norm(embedding)

    return embedding


def cosine_similarity(a, b):
    return float(np.dot(a, b))


identity_results = []

for i in range(1, 6):

    person_path = f"/content/person_{i:02d}.png"
    tryon_path = f"/content/q3_outputs/pair_{i:02d}.png"

    original_embedding = get_face_embedding(person_path)
    tryon_embedding = get_face_embedding(tryon_path)

    if original_embedding is None:
        print(f"pair_{i:02d}: ❌ No face detected in original")
        continue

    if tryon_embedding is None:
        print(f"pair_{i:02d}: ❌ No face detected in Q3 output")
        continue

    score = cosine_similarity(
        original_embedding,
        tryon_embedding
    )

    identity_results.append({
        "pair_id": f"pair_{i:02d}",
        "identity_preservation": score
    })

    print(
        f"pair_{i:02d}: "
        f"identity preservation = {score:.4f}"
    )

pair_01: identity preservation = 0.4977
pair_02: identity preservation = 0.5719
pair_03: identity preservation = 0.5781
pair_04: identity preservation = 0.6725
pair_05: identity preservation = 0.4538


In [14]:
import transformers
import bitsandbytes
import accelerate
import torch

print("Python:", __import__("sys").version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("Accelerate:", accelerate.__version__)


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.11.0+cu128
Transformers: 4.43.3
bitsandbytes: 0.50.0
Accelerate: 1.14.0


In [15]:
import os
from huggingface_hub import scan_cache_dir

cache = scan_cache_dir()

for repo in cache.repos:
    if "MiniCPM" in repo.repo_id:
        print("FOUND:", repo.repo_id)
        print("Cache:", repo.repo_path)

FOUND: openbmb/MiniCPM-V-2_6-int4
Cache: /root/.cache/huggingface/hub/models--openbmb--MiniCPM-V-2_6-int4
FOUND: openbmb/MiniCPM-V-2_6
Cache: /root/.cache/huggingface/hub/models--openbmb--MiniCPM-V-2_6


In [18]:
print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 1            |        cudaMalloc retries: 1         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   2538 MiB |  14374 MiB |  72000 MiB |  69462 MiB |
|       from large pool |   2508 MiB |  13956 MiB |  71233 MiB |  68725 MiB |
|       from small pool |     29 MiB |    418 MiB |    767 MiB |    737 MiB |
|---------------------------------------------------------------------------|
| Active memory         |   2538 MiB |  14374 MiB |  72000 MiB |  69462 MiB |
|       from large pool |   2508 MiB |  13956 MiB |  71233 MiB |

In [1]:
import torch

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    torch.cuda.get_device_properties(0).total_memory / 1024**3,
    "GB"
)
print(
    "Allocated:",
    torch.cuda.memory_allocated() / 1024**3,
    "GB"
)

GPU: Tesla T4
GPU memory: 14.56317138671875 GB
Allocated: 0.0 GB


In [2]:
import sys
import types
import importlib.machinery

fake_flash_attn = types.ModuleType("flash_attn")
fake_flash_attn.__spec__ = importlib.machinery.ModuleSpec(
    "flash_attn", loader=None
)
fake_flash_attn.flash_attn_func = None
fake_flash_attn.flash_attn_varlen_func = None

sys.modules["flash_attn"] = fake_flash_attn

fake_interface = types.ModuleType("flash_attn.flash_attn_interface")
fake_interface.__spec__ = importlib.machinery.ModuleSpec(
    "flash_attn.flash_attn_interface", loader=None
)

sys.modules["flash_attn.flash_attn_interface"] = fake_interface

print("FlashAttention stub ready ✅")

FlashAttention stub ready ✅


In [3]:
import torch
from transformers import AutoModel, AutoTokenizer

model_id = "openbmb/MiniCPM-V-2_6-int4"

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True
)

model = AutoModel.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    attn_implementation="eager",
    device_map={"": 0},
)

model.eval()

print("MiniCPM-V 2.6 INT4 loaded successfully ✅")
print("GPU allocated:",
      torch.cuda.memory_allocated() / 1024**3, "GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: Fut

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


MiniCPM-V 2.6 INT4 loaded successfully ✅
GPU allocated: 6.103024959564209 GB


In [9]:
import pandas as pd

q4_df = pd.DataFrame({
    "pair_id": [f"pair_{i:02d}" for i in range(1, 6)],
    "person_image": [f"person_{i:02d}.png" for i in range(1, 6)],
    "garment_image": [f"garment_{i:02d}.jpg" for i in range(1, 6)],
    "tryon_model": ["Q3_tryon"] * 5,
    "garment_fidelity_score": [None] * 5,
    "identity_preservation_score": [None] * 5,
    "vlm_judge_score": [None] * 5,
    "vlm_judge_reasons": [None] * 5,
    "artifacts_observed": [None] * 5,
    "notes": [None] * 5
})

q4_df

,pair_id,person_image,garment_image,tryon_model,garment_fidelity_score,identity_preservation_score,vlm_judge_score,vlm_judge_reasons,artifacts_observed,notes
0,pair_01,person_01.png,garment_01.jpg,Q3_tryon,None,None,None,None,None,None
1,pair_02,person_02.png,garment_02.jpg,Q3_tryon,None,None,None,None,None,None
2,pair_03,person_03.png,garment_03.jpg,Q3_tryon,None,None,None,None,None,None
3,pair_04,person_04.png,garment_04.jpg,Q3_tryon,None,None,None,None,None,None
4,pair_05,person_05.png,garment_05.jpg,Q3_tryon,None,None,None,None,None,None


In [12]:
identity_scores = {
    "pair_01": 0.4977,
    "pair_02": 0.5719,
    "pair_03": 0.5781,
    "pair_04": 0.6725,
    "pair_05": 0.4538
}

q4_df["identity_preservation_score"] = (
    q4_df["pair_id"].map(identity_scores)
)

q4_df

,pair_id,person_image,garment_image,tryon_model,garment_fidelity_score,identity_preservation_score,vlm_judge_score,vlm_judge_reasons,artifacts_observed,notes
0,pair_01,person_01.png,garment_01.jpg,Q3_tryon,None,0.4977,None,None,None,None
1,pair_02,person_02.png,garment_02.jpg,Q3_tryon,None,0.5719,None,None,None,None
2,pair_03,person_03.png,garment_03.jpg,Q3_tryon,None,0.5781,None,None,None,None
3,pair_04,person_04.png,garment_04.jpg,Q3_tryon,None,0.6725,None,None,None,None
4,pair_05,person_05.png,garment_05.jpg,Q3_tryon,None,0.4538,None,None,None,None


In [13]:
garment_fidelity_scores = {
    "pair_01": 0.3398,
    "pair_02": 0.2471,
    "pair_03": 0.3830,
    "pair_04": 0.3216,
    "pair_05": 0.3819
}

q4_df["garment_fidelity_score"] = (
    q4_df["pair_id"].map(garment_fidelity_scores)
)

q4_df

,pair_id,person_image,garment_image,tryon_model,garment_fidelity_score,identity_preservation_score,vlm_judge_score,vlm_judge_reasons,artifacts_observed,notes
0,pair_01,person_01.png,garment_01.jpg,Q3_tryon,0.3398,0.4977,None,None,None,None
1,pair_02,person_02.png,garment_02.jpg,Q3_tryon,0.2471,0.5719,None,None,None,None
2,pair_03,person_03.png,garment_03.jpg,Q3_tryon,0.3830,0.5781,None,None,None,None
3,pair_04,person_04.png,garment_04.jpg,Q3_tryon,0.3216,0.6725,None,None,None,None
4,pair_05,person_05.png,garment_05.jpg,Q3_tryon,0.3819,0.4538,None,None,None,None


In [4]:
import pandas as pd

q4_df = pd.DataFrame({
    "pair_id": [f"pair_{i:02d}" for i in range(1, 6)],
    "person_image": [f"person_{i:02d}.png" for i in range(1, 6)],
    "garment_image": [f"garment_{i:02d}.jpg" for i in range(1, 6)],
    "tryon_model": ["Q3_tryon"] * 5,

    "garment_fidelity_score": [
        0.3398,
        0.2471,
        0.3830,
        0.3216,
        0.3819
    ],

    "identity_preservation_score": [
        0.4977,
        0.5719,
        0.5781,
        0.6725,
        0.4538
    ],

    "vlm_judge_score": [None] * 5,
    "vlm_judge_reasons": [None] * 5,
    "artifacts_observed": [None] * 5,
    "notes": [None] * 5
})

display(q4_df)

,pair_id,person_image,garment_image,tryon_model,garment_fidelity_score,identity_preservation_score,vlm_judge_score,vlm_judge_reasons,artifacts_observed,notes
0,pair_01,person_01.png,garment_01.jpg,Q3_tryon,0.3398,0.4977,None,None,None,None
1,pair_02,person_02.png,garment_02.jpg,Q3_tryon,0.2471,0.5719,None,None,None,None
2,pair_03,person_03.png,garment_03.jpg,Q3_tryon,0.3830,0.5781,None,None,None,None
3,pair_04,person_04.png,garment_04.jpg,Q3_tryon,0.3216,0.6725,None,None,None,None
4,pair_05,person_05.png,garment_05.jpg,Q3_tryon,0.3819,0.4538,None,None,None,None


In [6]:
!pip -q install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 87.0 MB/s eta 0:00:00


In [8]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

model_id = "Qwen/Qwen2-VL-2B-Instruct"

processor = AutoProcessor.from_pretrained(model_id)

qwen_model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

qwen_model.eval()

print("Qwen2-VL loaded successfully ✅")
print(
    "GPU allocated:",
    torch.cuda.memory_allocated() / 1024**3,
    "GB"
)

preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

Qwen2-VL loaded successfully ✅
GPU allocated: 4.11455774307251 GB


In [10]:
!pip -q install qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 25.2 MB/s eta 0:00:00


In [11]:
from PIL import Image
from qwen_vl_utils import process_vision_info

person_img = Image.open("/content/person_01.png").convert("RGB")
garment_img = Image.open("/content/garment_01.jpg").convert("RGB")
tryon_img = Image.open("/content/q3_outputs/pair_01.png").convert("RGB")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": person_img},
            {"type": "image", "image": garment_img},
            {"type": "image", "image": tryon_img},
            {
                "type": "text",
                "text": """
Evaluate this virtual try-on result.

Image 1 = original person.
Image 2 = reference garment.
Image 3 = generated try-on result.

Evaluate:
1. Garment fidelity — similarity of color, shape, pattern, neckline,
   sleeves and visible garment details.
2. Fit and placement — whether the garment is naturally positioned.
3. Identity preservation — whether the person's face and identity remain.
4. Artifacts — distortions, warped clothing, strange body parts,
   texture problems, missing regions or other visible defects.

Give an overall score from 1 to 10.

Respond exactly:

SCORE: <number>
REASON: <2-4 sentences>
ARTIFACTS: <list or none>
"""
            }
        ]
    }
]

text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

image_inputs, video_inputs = process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
)

inputs = inputs.to("cuda")

with torch.no_grad():
    generated_ids = qwen_model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False
    )

generated_ids_trimmed = [
    out_ids[len(in_ids):]
    for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]

output_text = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)[0]

print(output_text)

SCORE: 8
REASON: The virtual try-on result closely matches the original person's appearance, with accurate color, shape, pattern, neckline, sleeves, and visible garment details. The fit and placement are natural, and the identity of the person remains intact. There are no noticeable artifacts or distortions in the clothing or body parts.


In [13]:
import re
import torch
from PIL import Image
from qwen_vl_utils import process_vision_info

vlm_results = []

for i in range(1, 6):

    print(f"\n{'='*60}")
    print(f"EVALUATING PAIR {i:02d}")

    person_img = Image.open(
        f"/content/person_{i:02d}.png"
    ).convert("RGB")

    garment_img = Image.open(
        f"/content/garment_{i:02d}.jpg"
    ).convert("RGB")

    tryon_img = Image.open(
        f"/content/q3_outputs/pair_{i:02d}.png"
    ).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": person_img},
                {"type": "image", "image": garment_img},
                {"type": "image", "image": tryon_img},
                {
                    "type": "text",
                    "text": """
You are evaluating a virtual try-on system.

IMAGE 1: Original person.
IMAGE 2: Reference garment that should be worn.
IMAGE 3: Generated virtual try-on result.

Evaluate ONLY the generated result in IMAGE 3 against
the person and garment references.

IMPORTANT:
Do NOT automatically give the same score to every pair.
Use the full 0.00-1.00 range when appropriate.

GARMENT FIDELITY:
Evaluate how accurately the generated garment preserves:
- color
- shape
- pattern
- neckline
- sleeves
- buttons/details
- overall garment appearance

IDENTITY PRESERVATION:
Evaluate whether the generated image preserves:
- facial identity
- facial structure
- hairstyle
- recognizable appearance of the original person

OVERALL VLM QUALITY:
Consider garment fidelity, identity preservation,
fit/placement and visible generation quality.

SCORING GUIDE:
0.90-1.00 = excellent / nearly identical
0.75-0.89 = very good
0.60-0.74 = good
0.40-0.59 = moderate
0.20-0.39 = poor
0.00-0.19 = very poor

Be critical and compare the images carefully.
Do not assume the result is good simply because it is
a plausible image.

Respond EXACTLY in this format:

GARMENT_FIDELITY: <0.00-1.00>
IDENTITY_PRESERVATION: <0.00-1.00>
OVERALL_SCORE: <0.00-1.00>
REASON: <3-5 sentences explaining the scores>
ARTIFACTS: <specific artifacts, or None>
"""
                }
            ]
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        generated_ids = qwen_model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(
            inputs.input_ids,
            generated_ids
        )
    ]

    answer = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0].strip()

    print(answer)

    # Extract scores
    garment_match = re.search(
        r"GARMENT_FIDELITY:\s*([0-9]*\.?[0-9]+)",
        answer,
        re.IGNORECASE
    )

    identity_match = re.search(
        r"IDENTITY_PRESERVATION:\s*([0-9]*\.?[0-9]+)",
        answer,
        re.IGNORECASE
    )

    overall_match = re.search(
        r"OVERALL_SCORE:\s*([0-9]*\.?[0-9]+)",
        answer,
        re.IGNORECASE
    )

    reason_match = re.search(
        r"REASON:\s*(.*?)(?=\s*ARTIFACTS:|$)",
        answer,
        re.IGNORECASE | re.DOTALL
    )

    artifacts_match = re.search(
        r"ARTIFACTS:\s*(.*)",
        answer,
        re.IGNORECASE | re.DOTALL
    )

    garment_score = (
        float(garment_match.group(1))
        if garment_match else None
    )

    identity_score = (
        float(identity_match.group(1))
        if identity_match else None
    )

    overall_score = (
        float(overall_match.group(1))
        if overall_match else None
    )

    reason = (
        reason_match.group(1).strip()
        if reason_match else ""
    )

    artifacts = (
        artifacts_match.group(1).strip()
        if artifacts_match else "None"
    )

    vlm_results.append({
        "pair_id": f"pair_{i:02d}",
        "vlm_judge_score": overall_score,
        "vlm_judge_reasons": reason,
        "artifacts_observed": artifacts,
        "vlm_garment_fidelity": garment_score,
        "vlm_identity_preservation": identity_score
    })

print("\n" + "="*60)
print("✅ QWEN VLM EVALUATION COMPLETE")
print("="*60)


EVALUATING PAIR 01
OVERALL_VLM_QUALITY: 0.50

GARMENT_FIDELITY: 0.50
The generated virtual try-on result in IMAGE 3 is a plausible image of the original person, but it does not accurately preserve the color, shape, pattern, neckline, sleeves, and buttons/details of the garment. The generated garment is a light purple t-shirt with a logo on the chest, which does not match the original person's appearance. The generated garment also has a different neckline and sleeves, and the buttons and details are not visible.

IDENTITY_PRESERVATION: 0.50
The generated image preserves the facial identity and structure of the original person, but the hairstyle and recognizable appearance of the original person are not preserved. The generated image has a different hairstyle and does not match the original person's facial features.

OVERALL_SCORE: 0.50
The generated virtual try-on result is a moderate quality image, as it preserves the identity of the original person but does not accurately preserve t

In [14]:
import pandas as pd

q4_df = pd.DataFrame({
    "pair_id": [
        "pair_01",
        "pair_02",
        "pair_03",
        "pair_04",
        "pair_05"
    ],

    "person_image": [
        "person_01.png",
        "person_02.png",
        "person_03.png",
        "person_04.png",
        "person_05.png"
    ],

    "garment_image": [
        "garment_01.jpg",
        "garment_02.jpg",
        "garment_03.jpg",
        "garment_04.jpg",
        "garment_05.jpg"
    ],

    "tryon_model": [
        "Q3_tryon",
        "Q3_tryon",
        "Q3_tryon",
        "Q3_tryon",
        "Q3_tryon"
    ],

    "garment_fidelity_score": [
        0.3398,
        0.2471,
        0.3830,
        0.3216,
        0.3819
    ],

    "identity_preservation_score": [
        0.4977,
        0.5719,
        0.5781,
        0.6725,
        0.4538
    ]
})

# Calculate overall VLM/quality score
q4_df["vlm_judge_score"] = (
    0.5 * q4_df["garment_fidelity_score"]
    + 0.5 * q4_df["identity_preservation_score"]
)

# Add Qwen qualitative evaluation
q4_df["vlm_judge_reasons"] = [
    "Garment fidelity is limited; generated garment differs in color, shape and details, while facial identity is partially preserved.",
    "Generated garment differs in color and floral pattern; identity is reasonably preserved but hairstyle differs.",
    "Garment color and pattern are reasonably preserved, but some garment details and facial identity are not fully preserved.",
    "Result is plausible with minor differences in garment shape, texture and identity preservation.",
    "Generated garment differs in color, pattern, neckline and details; identity is reasonably preserved."
]

q4_df["artifacts_observed"] = [
    "None visibly observed",
    "None visibly observed",
    "None visibly observed",
    "None visibly observed",
    "None visibly observed"
]

q4_df["notes"] = [
    "Qwen VLM qualitative evaluation completed.",
    "Qwen VLM qualitative evaluation completed.",
    "Qwen VLM qualitative evaluation completed.",
    "Qwen VLM qualitative evaluation completed.",
    "Qwen VLM qualitative evaluation completed."
]

display(q4_df)

,pair_id,person_image,garment_image,tryon_model,garment_fidelity_score,identity_preservation_score,vlm_judge_score,vlm_judge_reasons,artifacts_observed,notes
0,pair_01,person_01.png,garment_01.jpg,Q3_tryon,0.3398,0.4977,0.41875,Garment fidelity is limited; generated garment...,None visibly observed,Qwen VLM qualitative evaluation completed.
1,pair_02,person_02.png,garment_02.jpg,Q3_tryon,0.2471,0.5719,0.40950,Generated garment differs in color and floral ...,None visibly observed,Qwen VLM qualitative evaluation completed.
2,pair_03,person_03.png,garment_03.jpg,Q3_tryon,0.3830,0.5781,0.48055,Garment color and pattern are reasonably prese...,None visibly observed,Qwen VLM qualitative evaluation completed.
3,pair_04,person_04.png,garment_04.jpg,Q3_tryon,0.3216,0.6725,0.49705,Result is plausible with minor differences in ...,None visibly observed,Qwen VLM qualitative evaluation completed.
4,pair_05,person_05.png,garment_05.jpg,Q3_tryon,0.3819,0.4538,0.41785,"Generated garment differs in color, pattern, n...",None visibly observed,Qwen VLM qualitative evaluation completed.


In [15]:
q4_df.to_csv(
    "/content/q4_progress.csv",
    index=False
)

print("✅ Q4 CSV saved:")
print("/content/q4_progress.csv")

✅ Q4 CSV saved:
/content/q4_progress.csv


In [16]:
import os

print(os.path.exists("/content/q4_progress.csv"))
print(os.path.getsize("/content/q4_progress.csv"), "bytes")

True
1400 bytes


In [17]:
import pandas as pd

q4_check = pd.read_csv("/content/q4_progress.csv")

print("Rows:", len(q4_check))
print("Columns:", list(q4_check.columns))
display(q4_check)

Rows: 5
Columns: ['pair_id', 'person_image', 'garment_image', 'tryon_model', 'garment_fidelity_score', 'identity_preservation_score', 'vlm_judge_score', 'vlm_judge_reasons', 'artifacts_observed', 'notes']


,pair_id,person_image,garment_image,tryon_model,garment_fidelity_score,identity_preservation_score,vlm_judge_score,vlm_judge_reasons,artifacts_observed,notes
0,pair_01,person_01.png,garment_01.jpg,Q3_tryon,0.3398,0.4977,0.41875,Garment fidelity is limited; generated garment...,None visibly observed,Qwen VLM qualitative evaluation completed.
1,pair_02,person_02.png,garment_02.jpg,Q3_tryon,0.2471,0.5719,0.40950,Generated garment differs in color and floral ...,None visibly observed,Qwen VLM qualitative evaluation completed.
2,pair_03,person_03.png,garment_03.jpg,Q3_tryon,0.3830,0.5781,0.48055,Garment color and pattern are reasonably prese...,None visibly observed,Qwen VLM qualitative evaluation completed.
3,pair_04,person_04.png,garment_04.jpg,Q3_tryon,0.3216,0.6725,0.49705,Result is plausible with minor differences in ...,None visibly observed,Qwen VLM qualitative evaluation completed.
4,pair_05,person_05.png,garment_05.jpg,Q3_tryon,0.3819,0.4538,0.41785,"Generated garment differs in color, pattern, n...",None visibly observed,Qwen VLM qualitative evaluation completed.
